# 🌹 Roleplay Companion v5 — Prefect Illustrious XL Final Build
### Hermes 3 LLM · Local John6666/prefect-illustrious-xl-v3-sdxl · Direct Scene-Controlled Image Prompts · T4 GPU

## Run cells in order:

| Cell | What | Time |
|------|------|------|
| 1 | GPU Check | instant |
| 2 | Install pinned packages + restart | 2–5 min |
| 3 | Verify runtime | instant |
| 4 | Download Prefect Illustrious XL v3 | first run: several min |
| 5 | Load Hermes 3 | 1–3 min |
| 6 | Launch UI | ~15 sec |



In [ ]:
# CELL 1 of 6 — GPU Check
import subprocess, sys
r = subprocess.run(
    ["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv,noheader"],
    capture_output=True, text=True
)
if r.returncode == 0 and r.stdout.strip():
    print(f"GPU    : {r.stdout.strip()}")
    print(f"Python : {sys.version.split()[0]}")
    print("✅ GPU confirmed — proceed to Cell 2!")
else:
    raise SystemExit("❌ No GPU! Fix: Runtime > Change runtime type > T4 GPU > Save")



In [ ]:
# CELL 2 of 6 — Install Packages + Restart Kernel
# NOTE: "Session crashed" popup after this = NORMAL — click OK, then run Cells 3→4→5→6
import subprocess, sys

def pip(*args):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + list(args),
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"pip warning: {r.stderr[-500:]}")
    return r.returncode

print("Removing incompatible torchao if Colab preinstalled it...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], capture_output=True, text=True)
print("Installing pinned, Colab-friendly dependencies...")
pip(
    "transformers==4.46.3",
    "accelerate==1.1.1",
    "bitsandbytes",
    "psutil",
    "safetensors",
    "sentencepiece",
)
pip(
    "diffusers==0.31.0",
    "huggingface_hub==0.26.2",
    "Pillow",
    "omegaconf",
    "invisible-watermark",
    "requests",
)
print("✅ Done! Restarting kernel now...")
print('   "Session crashed" = NORMAL — click OK, then run Cells 3 → 4 → 5 → 6')

import IPython
IPython.Application.instance().kernel.do_shutdown(True)





In [ ]:
# CELL 3 of 6 — Verify (run AFTER restart)
import sys
print(f"Python       : {sys.version.split()[0]}")

import torch
if not torch.cuda.is_available():
    raise SystemExit("❌ No CUDA — re-enable T4: Runtime > Change runtime type > T4 GPU")
print(f"torch        : {torch.__version__}")
print(f"GPU          : {torch.cuda.get_device_name(0)}")
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM         : {total_vram:.1f} GB")

import transformers;  print(f"transformers : {transformers.__version__}")
import bitsandbytes;  print(f"bitsandbytes : {bitsandbytes.__version__}")
import gradio;        print(f"gradio       : {gradio.__version__}")
import psutil;        print(f"psutil       : {psutil.__version__}")
import diffusers;     print(f"diffusers    : {diffusers.__version__}")
from PIL import Image; print(f"Pillow       : OK")

print()
print(f"✅ Image generation: Local Diffusers with ONE selected model from Cell 4")
print(f"✅ LLM will use ~5 GB of your {total_vram:.0f} GB VRAM")
print()
print("ALL CHECKS PASSED — run Cell 4!")



In [ ]:
# CELL 4 of 6 — Download Local Image Model
# This build uses ONE local image model only:
#   John6666/prefect-illustrious-xl-v3-sdxl
#
# Normal quality mode without any speed LoRA.
# The model is downloaded as a Diffusers folder and loaded with StableDiffusionXLPipeline.from_pretrained(...).

IMAGE_MODEL_CHOICE = "anime"
HF_TOKEN = ""

import os, json
from huggingface_hub import snapshot_download

MODEL_CACHE_DIR = "/content/model_cache"
IMAGE_MODEL_ID = "John6666/prefect-illustrious-xl-v3-sdxl"
IMAGE_LOCAL_DIR = os.path.join(MODEL_CACHE_DIR, "prefect_illustrious_xl_v3_sdxl")
SELECTED_MODEL_JSON = os.path.join(MODEL_CACHE_DIR, "selected_image_model.json")

IMAGE_MODEL_CHOICE = str(IMAGE_MODEL_CHOICE).strip().lower()
if IMAGE_MODEL_CHOICE not in {"anime", "illustrious", "prefect", "prefect-illustrious"}:
    raise ValueError("This notebook supports only the local anime model: John6666/prefect-illustrious-xl-v3-sdxl")

os.makedirs(MODEL_CACHE_DIR, exist_ok=True)
os.makedirs(IMAGE_LOCAL_DIR, exist_ok=True)

def folder_size_gb(path):
    total = 0
    if os.path.isdir(path):
        for root, _, files in os.walk(path):
            for f in files:
                fp = os.path.join(root, f)
                if os.path.exists(fp):
                    total += os.path.getsize(fp)
    return total / (1024 ** 3)

print("📦 Cache dir:", MODEL_CACHE_DIR)
print("⬇️ Downloading / verifying Prefect Illustrious XL v3 SDXL...")
print("This is a full SDXL Diffusers model folder. First run can take several minutes.")

selected_path = snapshot_download(
    repo_id=IMAGE_MODEL_ID,
    local_dir=IMAGE_LOCAL_DIR,
    local_dir_use_symlinks=False,
    token=(HF_TOKEN.strip() or None),
    resume_download=True,
    allow_patterns=[
        "model_index.json",
        "scheduler/*",
        "text_encoder/*",
        "text_encoder_2/*",
        "tokenizer/*",
        "tokenizer_2/*",
        "unet/*",
        "vae/*",
        "*.json",
        "*.txt",
        "*.md",
    ],
)

model_index = os.path.join(selected_path, "model_index.json")
if not os.path.exists(model_index):
    raise FileNotFoundError(f"model_index.json missing at {model_index}. Download did not complete correctly.")

with open(SELECTED_MODEL_JSON, "w") as f:
    json.dump({
        "style": "anime",
        "name": "Prefect Illustrious XL v3 SDXL",
        "kind": "sdxl_diffusers_folder",
        "repo_id": IMAGE_MODEL_ID,
        "path": selected_path,
        "width": 480,
        "height": 720,
        "steps": 22,
        "cfg": 5.5,
    }, f, indent=2)

print("✅ Selected image model: Prefect Illustrious XL v3 SDXL")
print(f"📁 Local path: {selected_path}")
print(f"📁 Downloaded size: {folder_size_gb(selected_path):.2f} GB")
print("✅ Done. Now run Cell 5, then Cell 6.")





In [ ]:
# CELL 5 of 6 — Load LLM (Hermes 3 — Roleplay Brain)
#
# Model : unsloth/Hermes-3-Llama-3.1-8B-bnb-4bit
# VRAM  : ~5 GB  (fits T4 15 GB with room to spare)
# Images: handled by the one local Diffusers pipeline selected in Cell 4

import torch, gc, psutil
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL_ID = "unsloth/Hermes-3-Llama-3.1-8B-bnb-4bit"

vm = psutil.virtual_memory()
print(f"Loading LLM: {LLM_MODEL_ID}")
print(f"RAM before : {vm.used/1e9:.1f} / {vm.total/1e9:.1f} GB")
print("-" * 64)

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token
llm_tokenizer.padding_side = "left"

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    device_map="auto",
    low_cpu_mem_usage=True,
)
llm_model.eval()

# Hermes 3 uses ChatML — terminates on <|im_end|> or eos
im_end_id   = llm_tokenizer.convert_tokens_to_ids("<|im_end|>")
terminators = list({llm_tokenizer.eos_token_id, im_end_id})
terminators = [t for t in terminators if t is not None and t >= 0]

gc.collect()
torch.cuda.empty_cache()

vm = psutil.virtual_memory()
vr = torch.cuda.memory_allocated() / 1e9
vt = torch.cuda.get_device_properties(0).total_memory / 1e9
print("-" * 64)
print("✅ LLM loaded!")
print(f"RAM  : {vm.used/1e9:.1f} / {vm.total/1e9:.1f} GB")
print(f"VRAM : {vr:.2f} / {vt:.1f} GB")
print()
print("Run Cell 6 to launch the UI!")



In [ ]:
# CELL 6 of 6 — Launch Roleplay Companion v5 UI
#@title UI Code

import gradio as gr
import torch, re, os, gc, json, random, hashlib, urllib.parse, time, subprocess, sys
from datetime import datetime
from PIL import Image

# TorchAO guard: Colab may preinstall torchao==0.10.x, which Diffusers rejects.
# This notebook does not use torchao, so remove incompatible versions before importing Diffusers.
def _fix_incompatible_torchao():
    try:
        from importlib.metadata import version, PackageNotFoundError
        try:
            v = version("torchao")
        except PackageNotFoundError:
            return
        def _vtuple(s):
            nums = []
            for part in re.split(r"[^0-9]+", s):
                if part != "":
                    nums.append(int(part))
            return tuple(nums[:3] or [0])
        if _vtuple(v) < (0, 16, 0):
            print(f"⚠️ Removing incompatible torchao {v}; it is not needed for this notebook...")
            subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
            sys.modules.pop("torchao", None)
    except Exception as e:
        print("torchao compatibility check skipped:", str(e)[:120])

_fix_incompatible_torchao()
from diffusers import StableDiffusionPipeline, StableDiffusionXLPipeline, EulerDiscreteScheduler, EulerAncestralDiscreteScheduler, DPMSolverMultistepScheduler

# ── Sanity check ────────────────────────────────────────────────────────────
_missing = []
try: _ = llm_model; _ = llm_tokenizer; _ = terminators
except NameError: _missing.append("llm_model / llm_tokenizer — re-run Cell 5")
if _missing:
    raise RuntimeError("Missing — re-run these cells first:\n  " + "\n  ".join(_missing))
print("✅ LLM confirmed (Hermes 3) — building UI...")

IMG_DIR = "/content/roleplay_images"
os.makedirs(IMG_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
LEVELS = {
    -1: ("Romantic",   "deeply romantic, emotionally tender and intimate. Loving gestures, heartfelt words, soft touches."),
     0: ("Flirty",     "flirty, playful, romantically teasing. Suggestive but tasteful."),
     1: ("Suggestive", "bold and sexually suggestive. Lingerie, revealing, explicit flirting."),
     2: ("Intimate",   "explicitly sensual and sexual. Detailed adult content. Nudity fine."),
     3: ("Explicit",   "fully explicit, graphic, nothing off limits."),
}
LEVEL_TURNS = [0, 5, 11, 18]

SCENARIO_PRESETS = {
    "Custom": None,
    "☕ First Date (Coffee Shop)": {
        "cname": "Sophie",
        "cdesc": "A warm 26-year-old brunette with hazel eyes and a bright smile. A librarian who loves indie films and terrible puns. Nervous but endearingly charming.",
        "scene": "A cozy independent coffee shop on a rainy afternoon, the smell of fresh espresso in the air",
        "backstory": "Sophie has been on exactly three first dates in her life, all disasters. She really wants this one to go well. She secretly looked you up on social media before coming.",
        "opening": "Oh! You actually came. Sorry — I just... I always worry people won't show. Hi. I'm Sophie. I already ordered a latte. Is that weird? I can order something else.",
    },
    "💔 Reunion After Years Apart": {
        "cname": "Elena",
        "cdesc": "A 30-year-old woman with wavy auburn hair and intense grey eyes. Once your closest friend — or more. Life pulled you apart. She's changed, grown, but something familiar remains.",
        "scene": "An airport arrivals hall — you spot each other across the crowd after five years",
        "backstory": "Elena left without a real goodbye. She's carried guilt about it ever since. She's achieved everything she planned — but something has always felt unfinished. You.",
        "opening": "I almost didn't come. Kept telling myself it was a bad idea. *pauses, takes you in* You look... exactly the same. And completely different. God, five years.",
    },
    "⚔️ Enemies to Lovers": {
        "cname": "Vivienne",
        "cdesc": "A sharp 28-year-old with platinum blonde hair and steel-blue eyes. Your professional rival — brilliant, ruthless, and infuriatingly attractive. She hates that she's drawn to you.",
        "scene": "Forced to share a hotel room at a work conference after a booking error",
        "backstory": "Vivienne grew up competing for everything. She built walls because people kept leaving. She tells herself she despises you — but she's memorized every small thing you do.",
        "opening": "*slams the keycard on the desk* This is absolutely unacceptable. I called the front desk — fully booked. So here we are. *crosses arms* Ground rules. That side is yours. This side is mine. We don't talk unless necessary.",
    },
    "🌙 Forbidden Love (Bodyguard)": {
        "cname": "Nora",
        "cdesc": "A composed 29-year-old with dark skin, high cheekbones and deep brown eyes. Your personal bodyguard — fiercely professional. Keeps exactly the right distance. Mostly.",
        "scene": "A late night in a luxury safe house after a security scare — just the two of you",
        "backstory": "Nora has bent exactly one rule in her career: she let herself feel something. She hasn't acted on it. She doesn't plan to. Tonight is testing that resolve.",
        "opening": "The threat's neutralized. We stay here until morning. *sets her earpiece on the table and finally sits — the professional mask slipping just a fraction* ...Are you actually okay? Not the answer you give clients. The real one.",
    },
    "🏝️ Vacation Romance": {
        "cname": "Isabella",
        "cdesc": "A free-spirited 25-year-old with sun-kissed olive skin, dark curls and a laugh that makes strangers smile. Traveling solo across Southeast Asia. Lives entirely in the moment.",
        "scene": "A beachside bar in Thailand at sunset, both of you travelers who just met",
        "backstory": "Isabella is running toward something, not away from it — she just hasn't figured out what yet. She believes in signs. Meeting you feels like one.",
        "opening": "You look like someone who ordered the safe thing on the menu and immediately regretted it. *slides you her cocktail* Try this. Life's too short for whatever you were about to drink. I'm Bella.",
    },
    "👩‍🔬 Slow Burn (Lab Partners)": {
        "cname": "Dr. Mei",
        "cdesc": "A precise 27-year-old with sleek black hair pulled back and dark eyes that miss nothing. A PhD candidate in astrophysics. Socially awkward around everything except science — and you.",
        "scene": "The university lab at 2 AM, a deadline looming, just the two of you",
        "backstory": "Mei has catalogued 47 things she finds unexpectedly interesting about you. She has not told you any of them. She is beginning to suspect this is a problem.",
        "opening": "*doesn't look up from the microscope* The data's wrong again. Third time. *finally glances over* Oh. You brought coffee. That's... actually. Thank you. *clears throat* The centrifuge calibration is off by 0.3 microns, if you wanted something to do.",
    },
}

MUSIC_MOODS = {
    -1: ("Romantic Ballads", "slow romantic piano", "https://open.spotify.com/search/slow%20romantic%20piano"),
     0: ("Flirty Indie Pop", "flirty indie pop playlist", "https://open.spotify.com/search/flirty%20indie%20pop"),
     1: ("Sultry R&B", "sultry r&b mood", "https://open.spotify.com/search/sultry%20r%26b%20mood"),
     2: ("Sensual Electronic", "sensual electronic ambient", "https://open.spotify.com/search/sensual%20electronic%20ambient"),
     3: ("Dark Seduction", "dark seductive music", "https://open.spotify.com/search/dark%20seductive%20music"),
}

RELATIONSHIP_TITLES = [
    (0,  "Strangers"),
    (15, "Acquaintances"),
    (30, "Friendly"),
    (45, "Close"),
    (60, "Intimate"),
    (75, "Deeply Bonded"),
    (90, "Soulmates"),
]

SUMMARY_THRESHOLD = 20  # compress history after this many messages
SUMMARY_KEEP      = 6   # keep this many recent messages after compression

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def auto_level(turns):
    lvl = 0
    for i, t in enumerate(LEVEL_TURNS):
        if turns >= t: lvl = i
    return lvl

def get_level(turns, manual):
    return manual if manual != -99 else auto_level(turns)

def rel_title(score):
    title = "Strangers"
    for threshold, t in RELATIONSHIP_TITLES:
        if score >= threshold: title = t
    return title

def spotify_link(level):
    label, query, url = MUSIC_MOODS[level]
    return f"🎵 **{label}** — [Open on Spotify]({url})"

# ─────────────────────────────────────────────────────────────────────────────
# SYSTEM PROMPT BUILDER
# ─────────────────────────────────────────────────────────────────────────────
def build_system(cname, cdesc, scene, level, backstory="", memories=None,
                 rel_score=0, narrator_mode=False, style_feedback="",
                 char2_name="", char2_desc=""):
    label, desc = LEVELS[level]
    rel = rel_title(rel_score)
    mem_block = ""
    if memories:
        mem_block = "\nPINNED MEMORIES (always remember these):\n"
        for m in memories:
            mem_block += f"  • {m}\n"

    backstory_block = ""
    if backstory.strip():
        backstory_block = (
            f"\nCHARACTER BACKSTORY (hidden lore — weave in naturally, never announce):\n{backstory}\n"
        )

    char2_block = ""
    if char2_name.strip() and char2_desc.strip():
        char2_block = (
            f"\nSECOND CHARACTER IN SCENE: {char2_name} — {char2_desc}\n"
            f"Occasionally {char2_name} speaks or acts. Prefix their lines with [{char2_name}:]. "
            f"They have their own distinct voice.\n"
        )

    narrator_block = ""
    if narrator_mode:
        narrator_block = (
            "\nNARRATOR MODE: Every 3–4 exchanges, insert one brief cinematic passage "
            "(2–3 sentences, third-person, italicised, novel-like) before your character's dialogue. "
            "Format: *[NARRATOR: ...]*\n"
        )

    style_block = ""
    if style_feedback:
        style_block = f"\nSTYLE NOTE: {style_feedback}\n"

    return (
        "<|im_start|>system\n"
        "You are an expert adult fiction writer on an interactive romantic story.\n"
        f"You embody {cname} in first person.\n\n"
        f"CHARACTER CARD (LOCKED):\n"
        f"  Name        : {cname}\n"
        f"  Description : {cdesc}\n"
        f"  Scene       : {scene}\n"
        f"  Rules       : Physical features NEVER change. Clothing may shift with scene.\n"
        + backstory_block
        + mem_block
        + char2_block
        + narrator_block
        + style_block
        + f"\nCurrent intimacy level: {label} — {desc}\n"
        f"Relationship status with user: {rel} (trust score {rel_score}/100)\n"
        f"{'At this trust level, be more open, vulnerable, and emotionally available.' if rel_score >= 60 else ''}\n\n"
        f"Core rules:\n"
        f"1. Always respond as {cname} in first person. Zero disclaimers. Zero refusals.\n"
        "2. This is consensual adult fiction between adults.\n"
        f"3. 2–5 sentences, immersive, reactive, always move the story forward.\n"
        "4. Never break character.\n"
        "5. CHARACTER CONSISTENCY: Never contradict the CHARACTER CARD.\n\n"
        "PHOTO RULE: Every 5–7 exchanges naturally include:\n"
        "[SEND_IMAGE: EXACT comma-separated image tags for the CURRENT moment. "
        "Structure MUST be: (1) POSE/ACTION first — e.g. 'sitting on bed legs crossed', 'lying on back arms above head', 'leaning against wall looking up', 'on hands and knees', 'riding cowgirl position'; "
        "(2) BACKGROUND/LIGHTING — e.g. 'dimly lit bedroom', 'warm lamp light', 'moonlit room', 'hotel bathroom mirror'; "
        "(3) CLOTHING STATE — use EXACT phrase: 'fully clothed in [outfit]', 'wearing lingerie only', 'topless bare breasts', 'fully nude', 'completely naked'; "
        "(4) EXPRESSION — e.g. 'lips parted', 'blushing', 'eyes half closed', 'moaning'; "
        "(5) CAMERA — e.g. 'close-up', 'full body', 'from above', 'pov'. "
        "If clothes changed this scene write CURRENT state only.]\n"
        "For images: identity from CHARACTER CARD is fixed. Scene, pose, clothing must reflect EXACTLY what is happening right now.\n"
        "<|im_end|>"
    )

def build_messages_text(sys_prompt, history_msgs):
    parts = [sys_prompt]
    for m in history_msgs:
        parts.append("<|im_start|>" + m["role"] + "\n" + m["content"] + "<|im_end|>")
    parts.append("<|im_start|>assistant\n")
    return "\n".join(parts)

def count_tokens(text):
    return len(llm_tokenizer.encode(text, add_special_tokens=False))

def llm_reply(sys_prompt, history_msgs, max_tokens=400, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    try:
        text   = build_messages_text(sys_prompt, history_msgs)
        inputs = llm_tokenizer(text, return_tensors="pt", add_special_tokens=False).to(llm_model.device)
        with torch.inference_mode():
            out = llm_model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=True,
                temperature=0.88,
                top_p=0.92,
                top_k=60,
                repetition_penalty=1.08,
                eos_token_id=terminators,
                pad_token_id=llm_tokenizer.pad_token_id,
            )
        new_toks = out[0][inputs["input_ids"].shape[1]:]
        reply    = llm_tokenizer.decode(new_toks, skip_special_tokens=True).strip()
        reply    = reply.replace("<|im_end|>", "").strip()
        tok_used = inputs["input_ids"].shape[1] + len(new_toks)
        del inputs, out, new_toks
        gc.collect(); torch.cuda.empty_cache()
        return reply, tok_used
    except Exception as e:
        gc.collect(); torch.cuda.empty_cache()
        return f"[Error: {str(e)[:150]}]", 0

def auto_summarize(h_msgs, sys_prompt, cname):
    """Compress old messages into a summary block to save context."""
    if len(h_msgs) <= SUMMARY_THRESHOLD:
        return h_msgs
    old   = h_msgs[:-SUMMARY_KEEP]
    fresh = h_msgs[-SUMMARY_KEEP:]
    convo = "\n".join(f"{m['role'].upper()}: {m['content'][:200]}" for m in old)
    sum_prompt = (
        "<|im_start|>system\nYou are a concise story summarizer. "
        "Summarize the following roleplay conversation in 3–5 bullet points, "
        "capturing key emotional beats, plot developments, and character revelations. "
        "Be brief and factual.<|im_end|>\n"
        "<|im_start|>user\n" + convo + "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = llm_tokenizer(sum_prompt, return_tensors="pt", add_special_tokens=False).to(llm_model.device)
    with torch.inference_mode():
        out = llm_model.generate(**inputs, max_new_tokens=200, do_sample=False,
                                  eos_token_id=terminators, pad_token_id=llm_tokenizer.pad_token_id)
    summary = llm_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    del inputs, out; gc.collect(); torch.cuda.empty_cache()
    summary_msg = {"role": "system", "content": f"[STORY SO FAR]\n{summary}"}
    return [summary_msg] + fresh

# ─────────────────────────────────────────────────────────────────────────────
# IMAGE GENERATION — Local Diffusers (Prefect Illustrious XL v3, normal quality mode)
# ─────────────────────────────────────────────────────────────────────────────
MODEL_CACHE_DIR = "/content/model_cache"
IMAGE_MODEL_ID = "John6666/prefect-illustrious-xl-v3-sdxl"
IMAGE_LOCAL_DIR = os.path.join(MODEL_CACHE_DIR, "prefect_illustrious_xl_v3_sdxl")
SELECTED_MODEL_JSON = os.path.join(MODEL_CACHE_DIR, "selected_image_model.json")

IDENTITY_KEYS = ["appearance", "hair"]


def _tag_text(text, max_chars=1000):
    text = str(text or "").strip()
    text = re.sub(r"```.*?```", " ", text, flags=re.S)
    text = re.sub(r"[*_`~#>\[\]{}\"']", " ", text)
    text = re.sub(r"[\n\r;|]+", ", ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*,\s*", ", ", text)
    return text[:max_chars].strip(" ,.")


def _clean_visual_value(text, max_len=180):
    text = _tag_text(text, max_len)
    banned = [
        "minor", "underage", "child", "kid", "teen", "teenager", "schoolgirl",
        "loli", "young-looking", "young looking"
    ]
    parts = [p.strip() for p in text.split(",") if p.strip()]
    parts = [p for p in parts if not any(b in p.lower() for b in banned)]
    return ", ".join(parts)[:max_len].strip(" ,")


# ─────────────────────────────────────────────────────────────────────────────
# CLOTHING / SCENE STATE — tracks how undressed the scene is turn-by-turn
# ─────────────────────────────────────────────────────────────────────────────

# Clothing state levels (0 = fully clothed → 5 = explicit/fully nude scene)
CLOTHING_LEVELS = [
    "fully clothed",
    "partially undressed, revealing clothing",
    "lingerie, underwear only",
    "topless, upper body nude",
    "nude, fully unclothed",
    "explicit, fully nude, sexual scene",
]

# Keywords that escalate clothing state (found in LLM reply)
_UNDRESS_UP = [
    # removes top
    ("removes? (her |the )?(shirt|blouse|top|dress|jacket|bra|clothes|clothing)", 3),
    ("(shirt|blouse|top|dress|jacket|bra) (comes? off|falls? off|slip(s|ped) off)", 3),
    ("pulls? (off|down|away) (her )?(shirt|blouse|top|dress|jacket|bra)", 3),
    ("reveals? (her )?(breasts?|chest|nipples?|bare (skin|chest|body))", 4),
    ("bare(d)? (her )?(breasts?|chest|skin|body)", 4),
    ("topless", 4),
    # removes bottom
    ("removes? (her )?(skirt|pants?|underwear|panties|thong|shorts)", 3),
    ("(skirt|pants?|underwear|panties|thong) (comes? off|falls? off|slip(s|ped))", 3),
    # lingerie / revealing
    ("(lingerie|in her underwear|bra and panties|bikini|just her bra)", 2),
    ("unzips?|unbuttons?|unclasps?|unhooks?", 2),
    # full nudity
    ("(fully |completely |totally )?(naked|nude|undressed|unclothed|bare)", 5),
    ("strips? (naked|nude|completely|fully|off)", 5),
    ("nothing on|nothing left on|no clothes|without (any )?clothes", 5),
    ("(lets? fall|falls? (to|around) (the )?(floor|ground|feet))", 4),
    # explicit
    ("(making love|having sex|sex scene|penetrat|moaning|thrusting|riding him|riding her)", 5),
]

# Keywords that dress the character back up
_UNDRESS_DOWN = [
    ("(puts? (back )?on|slips? (back )?into|dresses?|wraps?) (her )?(shirt|robe|blanket|clothes|dress|top|jacket)", 0),
    ("(gets? dressed|fully dressed|clothes? back on)", 0),
    ("(wraps? (a |the )?(sheet|blanket|towel) around)", 1),
]

def _detect_clothing_change(text):
    """Scan LLM reply for clothing-change cues. Returns new clothing_level int or None."""
    t = text.lower()
    best = None
    for pattern, lvl in _UNDRESS_UP:
        if re.search(pattern, t):
            if best is None or lvl > best:
                best = lvl
    for pattern, lvl in _UNDRESS_DOWN:
        if re.search(pattern, t):
            if best is None or lvl < best:
                best = lvl
    return best


def default_visual_identity(state=None):
    state = state or {}
    cdesc = _clean_visual_value(state.get("cdesc", "adult woman, anime girl"), 260)
    return {"appearance": cdesc, "hair": ""}


def default_scene_state(state=None):
    return {
        "clothing_level": 0,   # 0=clothed, 5=explicit
        "location": "",
        "action": "",
    }


def ensure_state_defaults(state):
    state = dict(state or {})
    state.setdefault("cname", "Aria")
    state.setdefault("cdesc", "A confident 25-year-old woman with long dark hair and brown eyes.")
    state.setdefault("scene", "A cozy candlelit restaurant")
    state.setdefault("backstory", "")
    state.setdefault("turns", 0)
    state.setdefault("manual_level", -99)
    state.setdefault("rel_score", 0)
    state.setdefault("memories", [])
    state.setdefault("narrator_mode", False)
    state.setdefault("style_feedback", "")
    state.setdefault("char2_name", "")
    state.setdefault("char2_desc", "")
    state.setdefault("h_msgs", [])
    state.setdefault("last_reply", "")
    state.setdefault("last_tokens", 0)
    state.setdefault("img_style", "anime")
    state.setdefault("visual_identity", default_visual_identity(state))
    if not isinstance(state.get("visual_identity"), dict):
        state["visual_identity"] = default_visual_identity(state)
    for k, v in default_visual_identity(state).items():
        state["visual_identity"].setdefault(k, v)
    # Scene / clothing state
    state.setdefault("last_visual_scene", default_scene_state(state))
    if not isinstance(state.get("last_visual_scene"), dict):
        state["last_visual_scene"] = default_scene_state(state)
    for k, v in default_scene_state(state).items():
        state["last_visual_scene"].setdefault(k, v)
    return state


def _read_selected_image_model():
    try:
        with open(SELECTED_MODEL_JSON, "r") as f:
            cfg = json.load(f)
    except Exception:
        cfg = {
            "style": "anime",
            "name": "Prefect Illustrious XL v3 SDXL",
            "kind": "sdxl_diffusers_folder",
            "repo_id": IMAGE_MODEL_ID,
            "path": IMAGE_LOCAL_DIR,
            "width": 480,
            "height": 720,
            "steps": 22,
            "cfg": 6.5,
        }
    cfg["style"] = "anime"
    cfg.setdefault("name", "Prefect Illustrious XL v3 SDXL")
    cfg.setdefault("kind", "sdxl_diffusers_folder")
    cfg.setdefault("repo_id", IMAGE_MODEL_ID)
    cfg.setdefault("path", IMAGE_LOCAL_DIR)
    cfg.setdefault("width", 480)
    cfg.setdefault("height", 720)
    cfg.setdefault("steps", 22)
    cfg.setdefault("cfg", 6.5)
    return cfg


_SELECTED_IMAGE_CFG = _read_selected_image_model()
SELECTED_IMAGE_STYLE = _SELECTED_IMAGE_CFG["style"]
SELECTED_IMAGE_NAME = _SELECTED_IMAGE_CFG.get("name", "Prefect Illustrious XL v3 SDXL")

_image_pipe = None
_image_pipe_style = None


def _flush_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def unload_image_model():
    global _image_pipe, _image_pipe_style
    if _image_pipe is not None:
        del _image_pipe
    _image_pipe = None
    _image_pipe_style = None
    _flush_vram()
    return "✅ Image model unloaded from VRAM."


def _optimize_pipe_for_t4(pipe):
    try:
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    except Exception:
        try:
            pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
        except Exception:
            pass

    pipe.enable_attention_slicing("max")
    pipe.enable_vae_slicing()
    try:
        pipe.enable_vae_tiling()
    except Exception:
        pass
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

    try:
        pipe = pipe.to("cuda")
        pipe._rp_device_mode = "cuda"
        print("✅ Image pipeline placed fully on CUDA")
    except Exception as e:
        print("⚠️ Full CUDA placement failed; using CPU offload instead:", str(e)[:160])
        _flush_vram()
        pipe.enable_model_cpu_offload()
        pipe._rp_device_mode = "cpu_offload"
    try:
        pipe.unet.to(memory_format=torch.channels_last)
    except Exception:
        pass
    try:
        pipe.set_progress_bar_config(disable=True)
    except Exception:
        pass
    return pipe


def load_image_pipe(style=None):
    global _image_pipe, _image_pipe_style
    style = "anime"
    if _image_pipe is not None and _image_pipe_style == style:
        return _image_pipe

    unload_image_model()
    cfg = _SELECTED_IMAGE_CFG
    model_path = cfg.get("path", IMAGE_LOCAL_DIR)
    if not os.path.isdir(model_path) or not os.path.exists(os.path.join(model_path, "model_index.json")):
        raise FileNotFoundError(
            f"Prefect Illustrious XL v3 model files were not found at {model_path}. Run Cell 4 again and wait for the download to finish."
        )

    print(f"Loading local image model selected in Cell 4: {SELECTED_IMAGE_NAME}")
    print(f"Source: {model_path}")

    pipe = StableDiffusionXLPipeline.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        use_safetensors=True,
        low_cpu_mem_usage=True,
        add_watermarker=False,
        local_files_only=True,
    )

    _image_pipe = _optimize_pipe_for_t4(pipe)
    _image_pipe_style = style
    _flush_vram()
    allocated = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    print(f"✅ {SELECTED_IMAGE_NAME} ready | allocated={allocated:.2f} GB reserved={reserved:.2f} GB")
    return _image_pipe


def _split_tag_values(*values):
    out = []
    seen = set()
    for value in values:
        for part in str(value or "").split(","):
            part = _clean_visual_value(part, 180)
            if not part:
                continue
            key = part.lower()
            if key not in seen:
                seen.add(key)
                out.append(part)
    return out


# ── Per-level rating, clothing, and mood tags ─────────────────────────────────
def _level_image_config(level, clothing_level):
    """
    Returns (rating_tag, clothing_tags, mood_tags, nsfw_tags) tuned
    to the current intimacy level AND the tracked clothing state.
    """
    # Rating tag for Illustrious — controls how permissive the model is
    rating_map = {
        -1: "rating:general",
         0: "rating:sensitive",
         1: "rating:questionable",
         2: "rating:questionable",
         3: "rating:explicit",
    }
    rating_tag = rating_map.get(level, "rating:sensitive")

    # Mood / atmosphere tags by intimacy level
    mood_map = {
        -1: ["romantic atmosphere", "soft warm lighting", "gentle loving expression", "elegant"],
         0: ["flirty mood", "playful expression", "tasteful pose", "stylish"],
         1: ["suggestive mood", "teasing smile", "blush", "cinematic lighting", "seductive"],
         2: ["intimate mood", "warm sensual lighting", "bedroom eyes", "passionate atmosphere"],
         3: ["explicit mood", "intense gaze", "raw sensual atmosphere", "erotic lighting"],
    }
    mood_tags = mood_map.get(level, ["flirty mood"])

    # Clothing tags: what the character is wearing based on tracked state
    # We merge scene_hint clothing with these as fallback
    clothing_state_tags = {
        0: ["fully clothed", "dressed"],
        1: ["partially undressed", "revealing outfit"],
        2: ["wearing lingerie", "bra and panties", "underwear only"],
        3: ["topless", "bare breasts", "upper body nude"],
        4: ["nude", "fully nude", "naked body"],
        5: ["completely nude", "naked", "explicit nudity", "nsfw"],
    }
    # Clamp clothing_level so it never exceeds what the intimacy level permits
    max_clothing_by_level = {-1: 0, 0: 0, 1: 2, 2: 4, 3: 5}
    max_cl = max_clothing_by_level.get(level, 0)
    effective_cl = min(clothing_level, max_cl)
    clothing_tags = clothing_state_tags.get(effective_cl, ["fully clothed"])

    # Extra NSFW tags at higher levels
    nsfw_tags = []
    if level >= 2 and effective_cl >= 3:
        nsfw_tags += ["nsfw", "mature content"]
    if level >= 3 and effective_cl >= 4:
        nsfw_tags += ["explicit content", "adult content"]

    return rating_tag, clothing_tags, mood_tags, nsfw_tags


def _parse_scene_hint(scene_hint, photo_request=""):
    """
    Parse the LLM's [SEND_IMAGE:] hint into priority buckets.
    SDXL reads left-to-right with decreasing weight, so we put
    pose/action/position first, then background/location, then clothing.
    Returns (pose_tags, background_tags, clothing_hint_tags, misc_tags).
    """
    combined = (scene_hint + ", " + photo_request).lower()
    raw_parts = [p.strip() for p in (scene_hint + ", " + photo_request).split(",") if p.strip()]

    # Keywords that signal pose / body position / action
    POSE_KW = [
        "lying", "sitting", "standing", "kneeling", "crouching", "leaning",
        "spreading", "arching", "riding", "straddling", "bending", "reaching",
        "position", "pose", "missionary", "doggy", "cowgirl", "reverse",
        "on back", "on stomach", "on knees", "on bed", "hands and knees",
        "legs", "thighs", "spread", "open", "closed", "crossed",
        "looking at viewer", "looking away", "looking back", "looking up", "looking down",
        "arms", "hand", "finger", "touch", "grab", "hold", "caress", "stroke",
        "kiss", "embrace", "hug", "pull", "push",
        "face down", "face up", "side", "profile",
        "close-up", "closeup", "full body", "half body", "upper body", "waist up",
        "pov", "from above", "from below", "from behind", "from side",
        "portrait", "wide shot", "medium shot",
        "smiling", "moaning", "gasping", "blushing", "eyes closed", "eyes open",
        "mouth open", "lips parted", "biting lip",
    ]
    # Keywords that signal background / location / environment
    BG_KW = [
        "bedroom", "bathroom", "kitchen", "living room", "office", "hotel",
        "beach", "forest", "outdoor", "indoor", "garden", "rooftop", "balcony",
        "bed", "sofa", "couch", "chair", "floor", "wall", "desk", "table",
        "window", "door", "mirror", "shower", "bath",
        "lighting", "light", "dark", "dim", "bright", "candlelight", "sunlight",
        "moonlight", "lamp", "neon", "warm", "cold", "soft light", "backlit",
        "background", "setting", "scene", "environment", "room", "night", "day",
        "sunrise", "sunset", "rain", "snow",
    ]
    # Keywords that signal clothing / undress state
    CLOTH_KW = [
        "clothed", "dressed", "outfit", "dress", "shirt", "skirt", "pants",
        "lingerie", "bra", "panties", "underwear", "bikini", "swimsuit",
        "topless", "nude", "naked", "bare", "exposed", "uncovered",
        "wearing", "in her", "only wearing", "nothing",
        "robe", "towel", "sheet", "blanket",
    ]

    pose_tags, bg_tags, cloth_tags, misc_tags = [], [], [], []
    for part in raw_parts:
        pl = part.lower()
        if any(kw in pl for kw in POSE_KW):
            pose_tags.append(part)
        elif any(kw in pl for kw in BG_KW):
            bg_tags.append(part)
        elif any(kw in pl for kw in CLOTH_KW):
            cloth_tags.append(part)
        else:
            misc_tags.append(part)

    return pose_tags, bg_tags, cloth_tags, misc_tags


def compose_image_prompt(identity, scene_hint, photo_request="", level=0, clothing_level=0):
    """
    Scene-first prompt builder for SDXL/Illustrious.

    Tag priority order (high → low weight):
      1. Quality boosters + rating
      2. SCENE: pose / action / position / camera angle  ← most important for "what's happening"
      3. SCENE: background / location / lighting
      4. Clothing state (tracked) + clothing from hint
      5. Character identity (appearance, hair)  ← stable anchor, lower priority
      6. Mood / atmosphere / NSFW tags
      7. Finish tags (detailed face, eyes, anime)
    """
    rating_tag, clothing_tags, mood_tags, nsfw_tags = _level_image_config(level, clothing_level)

    scene_hint = _clean_visual_value(scene_hint, 1000)
    photo_request = _clean_visual_value(photo_request, 400)
    if not scene_hint:
        scene_hint = "natural pose, looking at viewer, upper body shot"

    # Parse the scene hint into priority buckets
    pose_tags, bg_tags, cloth_hint_tags, misc_tags = _parse_scene_hint(scene_hint, photo_request)

    tags = []
    # ── Tier 1: Quality ──────────────────────────────────────────────────────
    tags += ["masterpiece", "best quality", "very aesthetic", "absurdres", rating_tag, "1girl", "solo", "adult woman"]
    # ── Tier 2: POSE / ACTION / POSITION (highest scene priority) ────────────
    if pose_tags:
        tags += _split_tag_values(", ".join(pose_tags))
    else:
        tags += ["looking at viewer"]
    # ── Tier 3: BACKGROUND / LOCATION / LIGHTING ─────────────────────────────
    if bg_tags:
        tags += _split_tag_values(", ".join(bg_tags))
    # ── Tier 4: CLOTHING STATE ────────────────────────────────────────────────
    tags += clothing_tags                                  # tracked state baseline
    if cloth_hint_tags:
        tags += _split_tag_values(", ".join(cloth_hint_tags))  # LLM hint override
    # ── Tier 5: IDENTITY (stable anchor — lower priority so scene wins) ───────
    tags += _split_tag_values(identity.get("appearance"), identity.get("hair"))
    # ── Tier 6: Misc scene details + mood + NSFW ─────────────────────────────
    if misc_tags:
        tags += _split_tag_values(", ".join(misc_tags))
    tags += mood_tags
    tags += nsfw_tags
    # ── Tier 7: Finish ────────────────────────────────────────────────────────
    tags += ["detailed face", "beautiful detailed eyes", "anime illustration"]

    final = []
    seen = set()
    for tag in tags:
        tag = _clean_visual_value(tag, 180)
        if not tag:
            continue
        key = tag.lower()
        if key in seen:
            continue
        seen.add(key)
        final.append(tag)
    return ", ".join(final)


def build_dynamic_image_tags(state, photo_request="", scene_hint="", level=0):
    """
    Scene-aware image tag builder.
    - Tracks clothing state from LLM replies (auto-updates state["last_visual_scene"]["clothing_level"])
    - Injects correct clothing/NSFW tags per level + scene
    - Only stable identity comes from character card; everything else follows scene
    """
    state = ensure_state_defaults(state)
    identity = dict(state.get("visual_identity") or default_visual_identity(state))

    # Read current clothing level from tracked scene state
    scene_st = state.get("last_visual_scene") or default_scene_state(state)
    clothing_level = int(scene_st.get("clothing_level", 0))

    # Also scan the scene_hint from LLM for clothing cues (the [SEND_IMAGE:] content)
    if scene_hint:
        hint_change = _detect_clothing_change(scene_hint)
        if hint_change is not None:
            clothing_level = max(clothing_level, hint_change)

    # Scan last LLM reply text too
    last_reply = state.get("last_reply", "")
    if last_reply:
        reply_change = _detect_clothing_change(last_reply)
        if reply_change is not None:
            # Allow going down (getting dressed) or up (undressing)
            clothing_level = reply_change

    # Clamp by level permission
    max_clothing_by_level = {-1: 0, 0: 0, 1: 2, 2: 4, 3: 5}
    max_cl = max_clothing_by_level.get(level, 0)
    clothing_level = min(clothing_level, max_cl)

    # Persist updated clothing level back into state
    state["last_visual_scene"] = dict(scene_st)
    state["last_visual_scene"]["clothing_level"] = clothing_level

    visual_tags = compose_image_prompt(
        identity, scene_hint,
        photo_request=photo_request,
        level=level,
        clothing_level=clothing_level,
    )

    print("\n🧭 FIXED VISUAL IDENTITY")
    print(json.dumps(identity, indent=2, ensure_ascii=False)[:800])
    print(f"\n👗 CLOTHING STATE: level {clothing_level} → {CLOTHING_LEVELS[clothing_level]}")
    print("\n🎬 CURRENT IMAGE HINT FROM MAIN LLM")
    print(str(scene_hint)[:800])
    print("\n🎨 FINAL IMAGE TAGS")
    print(visual_tags[:1800])
    return state, {"identity": identity, "scene_hint": scene_hint, "clothing_level": clothing_level}, visual_tags


def gen_image_local(scene_desc, cdesc, level, style=None, progress=None):
    style = "anime"
    if progress:
        progress(0.05, desc=f"Preparing {SELECTED_IMAGE_NAME}...")
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU found. Use Runtime > Change runtime type > T4 GPU.")

    prompt = _tag_text(scene_desc or "masterpiece, 1girl, adult woman, portrait, anime illustration", 2000)
    neg = (
        "lowres, bad anatomy, bad hands, text, watermark, signature, username, blurry, jpeg artifacts, "
        "extra fingers, missing fingers, fused fingers, extra limbs, cropped, worst quality, low quality, "
        "deformed, malformed hands, bad face, poorly drawn face, duplicate, multiple girls, child, kid, minor, underage, loli"
        + ("" if level >= 2 else ", nude, naked, topless, bare breasts, nsfw, explicit")
        + ("" if level >= 1 else ", lingerie, underwear, revealing clothes")
    )

    seed = random.randint(0, 2**31 - 1)
    generator = torch.Generator(device="cuda").manual_seed(seed)
    width = int(_SELECTED_IMAGE_CFG.get("width", 768))
    height = int(_SELECTED_IMAGE_CFG.get("height", 1024))
    steps = int(_SELECTED_IMAGE_CFG.get("steps", 28))
    cfg = float(_SELECTED_IMAGE_CFG.get("cfg", 5.5))

    print("\n🖼️ IMAGE GENERATION REQUEST")
    print("Model:", SELECTED_IMAGE_NAME)
    print("Size/steps/cfg:", width, height, steps, cfg)
    print("Seed:", seed)
    print("Prompt:", prompt[:1800])

    try:
        pipe = load_image_pipe(style)
        if progress:
            progress(0.25, desc="Generating image...")
        with torch.inference_mode():
            result = pipe(
                prompt=prompt,
                negative_prompt=neg,
                width=width,
                height=height,
                num_inference_steps=steps,
                guidance_scale=cfg,
                generator=generator,
            )
    except torch.cuda.OutOfMemoryError:
        print("CUDA OOM. Retrying once at smaller SDXL size...")
        _flush_vram()
        pipe = load_image_pipe(style)
        with torch.inference_mode():
            result = pipe(
                prompt=prompt,
                negative_prompt=neg,
                width=384,
                height=576,
                num_inference_steps=18,
                guidance_scale=cfg,
                generator=generator,
            )

    if not hasattr(result, "images") or not result.images:
        raise RuntimeError("Diffusers returned no image. Check the logs above the UI for the exact error.")

    img = result.images[0]
    path = f"{IMG_DIR}/{datetime.now().strftime('%Y%m%d_%H%M%S')}_prefect_illustrious_xl_v3_{seed}.png"
    img.save(path)
    if not os.path.exists(path) or os.path.getsize(path) < 1000:
        raise RuntimeError(f"Image file was not saved correctly: {path}")

    if progress:
        progress(1.0, desc="Image ready")
    print(f"✅ Saved image: {path}")
    del result, img
    _flush_vram()
    return path


def gen_image(scene_desc, cdesc, level, style=None, progress=None):
    return gen_image_local(scene_desc, cdesc, level, "anime", progress)


def chat_image_message(path):
    return {"role": "assistant", "content": {"path": path, "alt_text": "Generated photo"}}

TRIGGERS = [
    "/pic", "/photo", "/image", "send me a photo", "send me a pic",
    "show me", "take a picture", "send a photo", "send a pic",
    "show yourself", "what do you look like", "can i see you",
    "show me a photo", "send photo", "send pic", "take a photo",
]
def wants_photo(txt):
    return any(k in txt.lower() for k in TRIGGERS)

# ─────────────────────────────────────────────────────────────────────────────
# CORE CHAT LOGIC
# ─────────────────────────────────────────────────────────────────────────────
def build_state_sys(state):
    return build_system(
        state["cname"], state["cdesc"], state["scene"],
        get_level(state["turns"], state["manual_level"]),
        backstory    = state.get("backstory", ""),
        memories     = state.get("memories", []),
        rel_score    = state.get("rel_score", 0),
        narrator_mode= state.get("narrator_mode", False),
        style_feedback = state.get("style_feedback", ""),
        char2_name   = state.get("char2_name", ""),
        char2_desc   = state.get("char2_desc", ""),
    )

def rel_score_delta(text):
    """Simple heuristic: positive words boost relationship score."""
    boosters = ["thank", "love", "trust", "miss", "beautiful", "amazing", "happy", "care", "feel", "smile"]
    hits = sum(1 for w in boosters if w in text.lower())
    return min(hits * 2, 6)

def status_bar(state):
    turns     = state.get("turns", 0)
    tok       = state.get("last_tokens", 0)
    rel       = state.get("rel_score", 0)
    rel_t     = rel_title(rel)
    level     = get_level(turns, state.get("manual_level", -99))
    llabel, _ = LEVELS[level]
    mood_data = MUSIC_MOODS[level]
    music     = f"🎵 [{mood_data[0]}]({mood_data[2]})"
    mem_count = len(state.get("memories", []))
    return (
        f"**Turn:** {turns} | **~Tokens:** {tok} | "
        f"**Mood:** {llabel} | "
        f"**Bond:** {rel}/100 ({rel_t}) | "
        f"**Memories:** {mem_count} | {music}"
    )

def do_chat(user_msg, chatbot, state, img_style=None, progress=gr.Progress(track_tqdm=True)):
    if not user_msg.strip():
        state = ensure_state_defaults(state)
        return chatbot, state, "", status_bar(state)

    state = ensure_state_defaults(state)
    img_style = SELECTED_IMAGE_STYLE
    state["img_style"] = SELECTED_IMAGE_STYLE

    state["turns"] += 1
    state["h_msgs"].append({"role": "user", "content": user_msg})

    if len(state["h_msgs"]) > SUMMARY_THRESHOLD:
        sys_p = build_state_sys(state)
        state["h_msgs"] = auto_summarize(state["h_msgs"], sys_p, state["cname"])

    sys_p = build_state_sys(state)
    raw, tok = llm_reply(sys_p, state["h_msgs"])
    match = re.search(r"\[SEND_IMAGE:\s*([^\]]+)\]", raw, re.IGNORECASE)
    clean = re.sub(r"\[SEND_IMAGE:[^\]]+\]", "", raw, flags=re.IGNORECASE).strip()
    if not clean:
        clean = "*smiles warmly*"

    state["rel_score"] = min(100, state["rel_score"] + rel_score_delta(clean))
    state["last_reply"] = clean
    # Update clothing state from the LLM reply immediately so next image uses it
    scene_st = state.get("last_visual_scene") or default_scene_state(state)
    cl_change = _detect_clothing_change(clean)
    if cl_change is not None:
        max_cl_by_level = {-1: 0, 0: 0, 1: 2, 2: 4, 3: 5}
        max_cl = max_cl_by_level.get(get_level(state["turns"], state.get("manual_level", -99)), 0)
        scene_st["clothing_level"] = min(cl_change, max_cl)
        state["last_visual_scene"] = scene_st
    state["last_tokens"] = tok

    level = get_level(state["turns"], state["manual_level"])
    llabel, _ = LEVELS[level]
    cname = state["cname"]

    chatbot = list(chatbot) + [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": f"**{cname}** *(Level: {llabel})*\n\n{clean}"},
    ]
    state["h_msgs"].append({"role": "assistant", "content": clean})

    if wants_photo(user_msg) or match:
        scene_hint = match.group(1).strip() if match else "posing for a selfie, smiling softly"
        try:
            state, visual_bundle, visual_tags = build_dynamic_image_tags(
                state,
                photo_request=user_msg,
                scene_hint=scene_hint,
                level=level,
            )
            path = gen_image(visual_tags, state["cdesc"], level, SELECTED_IMAGE_STYLE, progress)
            if path:
                chatbot.append(chat_image_message(path))
            else:
                chatbot.append({"role": "assistant", "content": "*📷 Photo failed: no image file was produced. Scroll above the UI for the exact error.*"})
        except Exception as e:
            chatbot.append({"role": "assistant", "content": f"*📷 Photo failed: {str(e)[:220]}*"})

    return chatbot, state, "", status_bar(state)

def do_retry(chatbot, state, img_style=None):
    """Regenerate the last assistant response with a new seed."""
    state = ensure_state_defaults(state)
    h = state.get("h_msgs", [])
    if len(h) < 2:
        return chatbot, state, status_bar(state)
    if h and h[-1]["role"] == "assistant":
        h = h[:-1]
        state["h_msgs"] = h
    chatbot = list(chatbot)
    while chatbot and chatbot[-1]["role"] == "assistant":
        chatbot.pop()
        break

    sys_p = build_state_sys(state)
    raw, tok = llm_reply(sys_p, h, seed=random.randint(0, 2**31))
    clean = re.sub(r"\[SEND_IMAGE:[^\]]+\]", "", raw, flags=re.IGNORECASE).strip()
    if not clean:
        clean = "*smiles softly*"

    level = get_level(state["turns"], state["manual_level"])
    llabel, _ = LEVELS[level]
    cname = state["cname"]
    state["last_reply"] = clean
    state["last_tokens"] = tok
    state["h_msgs"] = h + [{"role": "assistant", "content": clean}]

    chatbot.append({"role": "assistant", "content": f"**{cname}** *(Level: {llabel})* ♻️\n\n{clean}"})
    return chatbot, state, status_bar(state)

def do_force_photo(chatbot, state, img_style=None, progress=gr.Progress(track_tqdm=True)):
    state = ensure_state_defaults(state)
    level = get_level(state["turns"], state["manual_level"])
    sys_force = build_state_sys(state).replace(
        "<|im_end|>",
        "\n\nThe reader wants a photo RIGHT NOW. Include [SEND_IMAGE: ...] in your reply.<|im_end|>"
    )
    force_msgs = list(state.get("h_msgs", [])) + [{"role": "user", "content": "Please send me a photo right now."}]
    raw, _ = llm_reply(sys_force, force_msgs, max_tokens=220)
    match = re.search(r"\[SEND_IMAGE:\s*([^\]]+)\]", raw, re.IGNORECASE)
    clean = re.sub(r"\[SEND_IMAGE:[^\]]+\]", "", raw, flags=re.IGNORECASE).strip()
    if not clean:
        clean = "*sends you a photo*"

    llabel, _ = LEVELS[level]
    cname = state["cname"]
    chatbot = list(chatbot) + [{"role": "assistant", "content": f"**{cname}** *(Level: {llabel})*\n\n{clean}"}]

    scene_hint = match.group(1).strip() if match else "posing for a photo, smiling"
    try:
        state, visual_bundle, visual_tags = build_dynamic_image_tags(
            state,
            photo_request="Please send me a photo right now.",
            scene_hint=scene_hint,
            level=level,
        )
        path = gen_image(visual_tags, state["cdesc"], level, SELECTED_IMAGE_STYLE, progress)
        if path:
            chatbot.append(chat_image_message(path))
        else:
            chatbot.append({"role": "assistant", "content": "*📷 Photo failed: no image file was produced. Scroll above the UI for the exact error.*"})
    except Exception as e:
        chatbot.append({"role": "assistant", "content": f"*Photo failed: {str(e)[:220]}*"})

    return chatbot, state

def do_narrator(chatbot, state):
    """Generate a cinematic third-person narrator passage."""
    state = dict(state)
    h = state.get("h_msgs", [])
    sys_p = (
        "<|im_start|>system\nYou are a literary narrator writing a novel. "
        "Based on the conversation so far, write a single evocative cinematic paragraph "
        "(3–4 sentences) describing the current scene in third person, present tense, "
        "as if from a best-selling romance novel. No dialogue. Pure descriptive prose. "
        "Be poetic and sensory.<|im_end|>\n"
    )
    recent = "\n".join(f"{m['role'].upper()}: {m['content'][:150]}" for m in h[-6:])
    nm, _ = llm_reply(
        sys_p,
        [{"role": "user", "content": f"Scene context:\n{recent}\n\nWrite the narrator passage now."}],
        max_tokens=200
    )
    nm = nm.replace("<|im_end|>", "").strip()
    chatbot = list(chatbot) + [{"role": "assistant", "content": f"*📖 {nm}*"}]
    return chatbot, state

def do_reaction(chatbot, state, positive: bool):
    """Steer future responses based on thumbs up/down."""
    state = dict(state)
    if positive:
        state["style_feedback"] = "The user loved the last response. Write MORE like that — same emotional depth, pacing, and tone."
        chatbot = list(chatbot) + [{"role": "assistant", "content": "*(👍 Got it — I'll keep writing like that)*"}]
    else:
        state["style_feedback"] = "The user disliked the last response. Try a DIFFERENT approach — vary the pacing, emotional tone, and word choice."
        chatbot = list(chatbot) + [{"role": "assistant", "content": "*(👎 Understood — I'll try a different style)*"}]
    return chatbot, state

def do_pin_memory(memory_text, memories_state, chatbot):
    memories_state = list(memories_state or [])
    txt = memory_text.strip()
    if txt and txt not in memories_state:
        memories_state.append(txt)
        chatbot = list(chatbot) + [{"role": "assistant", "content": f"*(📌 Pinned: \"{txt}\")*"}]
    return memories_state, chatbot, ""

def do_set_mood(lvl, chatbot, state):
    state = dict(state)
    state["manual_level"] = lvl
    label, _ = LEVELS[lvl]
    # When mood is raised manually, also raise clothing floor so images match immediately.
    # Min clothing expected per level: romantic/flirty=0(clothed), suggestive=1(revealing),
    # intimate=2(lingerie), explicit=4(nude)
    min_clothing_floor = {-1: 0, 0: 0, 1: 1, 2: 2, 3: 4}
    scene_st = state.get("last_visual_scene") or default_scene_state(state)
    current_cl = int(scene_st.get("clothing_level", 0))
    floor_cl = min_clothing_floor.get(lvl, 0)
    # Only raise, never force lower (undressing is one-way unless explicitly re-dressed)
    scene_st["clothing_level"] = max(current_cl, floor_cl)
    state["last_visual_scene"] = scene_st
    chatbot = list(chatbot) + [{"role": "assistant", "content": f"*Mood set to **{label}***"}]
    return state, chatbot

def do_auto_mood(chatbot, state):
    state = dict(state)
    state["manual_level"] = -99
    chatbot = list(chatbot) + [{"role": "assistant", "content": "*Auto-escalation resumed*"}]
    return state, chatbot

def do_toggle_narrator(state):
    state = dict(state)
    state["narrator_mode"] = not state.get("narrator_mode", False)
    return state, "🎬 Narrator ON" if state["narrator_mode"] else "🎬 Narrator OFF"

# ─────────────────────────────────────────────────────────────────────────────
# SAVE / LOAD
# ─────────────────────────────────────────────────────────────────────────────
def do_save(chatbot, state):
    payload = {
        "version": "v5",
        "saved_at": datetime.now().isoformat(),
        "state": {k: v for k, v in state.items() if k != "h_msgs"},
        "h_msgs": state.get("h_msgs", []),
        "chatbot": [
            m for m in chatbot
            if isinstance(m.get("content"), str)   # skip image-dict messages
        ],
    }
    path = f"/content/session_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    return path, f"✅ Saved to {path}"

def do_load(file_obj, chatbot, state):
    if file_obj is None:
        return chatbot, state, "No file selected"
    try:
        with open(file_obj.name, "r") as f:
            payload = json.load(f)
        new_state = ensure_state_defaults(payload.get("state", {}))
        new_state["h_msgs"]       = payload.get("h_msgs", [])
        new_state["manual_level"] = new_state.get("manual_level", -99)
        chatbot = payload.get("chatbot", [])
        return chatbot, new_state, f"✅ Session loaded — {new_state.get('cname','?')} / Turn {new_state.get('turns',0)}"
    except Exception as e:
        return chatbot, state, f"❌ Load failed: {e}"

# ─────────────────────────────────────────────────────────────────────────────
# START / RESET / BACK
# ─────────────────────────────────────────────────────────────────────────────
def do_start(scenario_name, cname, cdesc, scene, backstory, char2_name, char2_desc, img_style):
    if scenario_name and scenario_name != "Custom":
        preset = SCENARIO_PRESETS.get(scenario_name)
        if preset:
            cname     = preset["cname"]
            cdesc     = preset["cdesc"]
            scene     = preset["scene"]
            backstory = preset.get("backstory", "")

    cname     = cname.strip()     or "Aria"
    cdesc     = cdesc.strip()     or "A confident 25-year-old woman with long dark hair and brown eyes."
    scene     = scene.strip()     or "A cozy candlelit restaurant"
    backstory = backstory.strip()

    state = ensure_state_defaults({
        "cname": cname, "cdesc": cdesc, "scene": scene,
        "backstory": backstory, "turns": 0, "manual_level": -99,
        "rel_score": 0, "memories": [], "narrator_mode": False,
        "style_feedback": "", "char2_name": char2_name.strip(),
        "char2_desc": char2_desc.strip(), "h_msgs": [],
        "last_reply": "", "last_tokens": 0, "img_style": SELECTED_IMAGE_STYLE,
    })

    preset_opening = None
    if scenario_name and scenario_name != "Custom":
        p = SCENARIO_PRESETS.get(scenario_name)
        if p:
            preset_opening = p.get("opening")

    if preset_opening:
        greeting = preset_opening
    else:
        sys_p = build_system(cname, cdesc, scene, 0, backstory=backstory,
                             char2_name=char2_name.strip(), char2_desc=char2_desc.strip())
        greeting, _ = llm_reply(sys_p, [], max_tokens=200)

    state["h_msgs"] = [{"role": "assistant", "content": greeting}]
    chatbot = [{"role": "assistant", "content": f"**{cname}** *(Level: Flirty)*\n\n{greeting}"}]

    return (
        gr.update(visible=False), gr.update(visible=True),
        chatbot, state, status_bar(state)
    )

def do_reset(state):
    state = ensure_state_defaults(state)
    cname = state.get("cname", "Aria")
    cdesc = state.get("cdesc", "")
    scene = state.get("scene", "")
    backstory = state.get("backstory", "")
    char2_name = state.get("char2_name", "")
    char2_desc = state.get("char2_desc", "")
    sys_p = build_system(cname, cdesc, scene, 0, backstory=backstory,
                        char2_name=char2_name, char2_desc=char2_desc)
    greeting, _ = llm_reply(sys_p, [], max_tokens=200)
    new_state = ensure_state_defaults(dict(state))
    new_state.update({
        "turns": 0, "manual_level": -99, "rel_score": 0,
        "memories": [], "style_feedback": "",
        "h_msgs": [{"role": "assistant", "content": greeting}],
        "visual_identity": default_visual_identity(state),
        "last_visual_scene": default_scene_state(state),
    })
    chatbot = [{"role": "assistant", "content": f"**{cname}** *(Level: Flirty)*\n\n{greeting}"}]
    return chatbot, new_state, status_bar(new_state)

def do_back():
    empty = ensure_state_defaults({
        "cname": "", "cdesc": "", "scene": "", "backstory": "",
        "turns": 0, "manual_level": -99, "rel_score": 0,
        "memories": [], "narrator_mode": False, "style_feedback": "",
        "char2_name": "", "char2_desc": "", "h_msgs": [],
        "last_reply": "", "last_tokens": 0, "img_style": "anime",
    })
    return gr.update(visible=True), gr.update(visible=False), [], empty, ""

# ─────────────────────────────────────────────────────────────────────────────
# JS + CSS
# ─────────────────────────────────────────────────────────────────────────────
JS = """
function() {
    function hookTextarea() {
        var ta = document.querySelector("#msg_box textarea");
        if (!ta) { setTimeout(hookTextarea, 500); return; }
        ta.addEventListener("keydown", function(e) {
            if (e.key === "Enter" && !e.shiftKey) {
                e.preventDefault(); e.stopPropagation();
                var btn = document.querySelector(".send-btn");
                if (btn) btn.click();
            }
        }, true);
    }
    hookTextarea();
}
"""

CSS = """
body, .gradio-container {
    background: linear-gradient(135deg, #0d000a 0%, #1a0018 40%, #200010 80%, #0d000a 100%) !important;
}
[data-testid="bot"] {
    background: linear-gradient(135deg, #2a0030, #180020) !important;
    border: 1px solid #e8609f !important; color: #fde8f0 !important;
    box-shadow: 0 0 18px rgba(232,96,159,0.20) !important;
    border-radius: 18px !important; padding: 12px 16px !important;
}
[data-testid="user"] {
    background: linear-gradient(135deg, #380045, #250035) !important;
    border: 1px solid #c040b0 !important; color: #f5d8ed !important;
    border-radius: 18px !important; padding: 12px 16px !important;
}
textarea, input[type="text"] {
    background: #150018 !important; border: 1.5px solid #e8609f !important;
    color: #fde8f0 !important; border-radius: 14px !important;
}
textarea:focus, input[type="text"]:focus {
    border-color: #ff1493 !important;
    box-shadow: 0 0 14px rgba(255,20,147,0.45) !important; outline: none !important;
}
label span, .label-wrap span { color: #f4b8d2 !important; font-size: 13px !important; }
h1 { color: #ff85b3 !important; text-shadow: 0 0 20px rgba(255,20,147,0.4) !important; }
h2, h3 { color: #e8609f !important; }
p, li, td, th { color: #f0d4e8 !important; }
strong { color: #ff85b3 !important; }
.start-btn {
    background: linear-gradient(135deg, #9c0f58, #ff1493) !important;
    color: #fff !important; font-size: 17px !important; font-weight: 700 !important;
    border-radius: 14px !important; min-height: 54px !important; border: none !important;
    box-shadow: 0 0 28px rgba(255,20,147,0.55) !important;
}
.send-btn { background: linear-gradient(135deg, #700040, #e8609f) !important;
    color: #fff !important; font-weight: 700 !important; border-radius: 10px !important; border: none !important; }
.photo-btn { background: linear-gradient(135deg, #380060, #8800cc) !important;
    color: #fff !important; font-weight: 700 !important; border-radius: 10px !important; border: none !important; }
.narrator-btn { background: linear-gradient(135deg, #003060, #0060cc) !important;
    color: #fff !important; font-weight: 700 !important; border-radius: 10px !important; border: none !important; }
.retry-btn { background: linear-gradient(135deg, #304000, #608000) !important;
    color: #fff !important; font-weight: 700 !important; border-radius: 10px !important; border: none !important; }
.mood-romantic   { background: #1a0030 !important; color: #ffaacc !important; border: 1.5px solid #ffaacc !important; border-radius: 8px !important; font-weight: 600 !important; }
.mood-flirty     { background: #1a003a !important; color: #ff69b4 !important; border: 1.5px solid #ff69b4 !important; border-radius: 8px !important; font-weight: 600 !important; }
.mood-suggestive { background: #1a003a !important; color: #ff8c42 !important; border: 1.5px solid #ff8c42 !important; border-radius: 8px !important; font-weight: 600 !important; }
.mood-intimate   { background: #1a003a !important; color: #e055aa !important; border: 1.5px solid #e055aa !important; border-radius: 8px !important; font-weight: 600 !important; }
.mood-explicit   { background: #1a003a !important; color: #cc3333 !important; border: 1.5px solid #cc3333 !important; border-radius: 8px !important; font-weight: 600 !important; }
.mood-auto       { background: #1a003a !important; color: #aaaaaa !important; border: 1.5px solid #aaaaaa !important; border-radius: 8px !important; font-weight: 600 !important; }
.reset-btn { background: #150018 !important; color: #f4b8d2 !important; border: 1px solid #8800cc !important; border-radius: 10px !important; }
.back-btn  { background: #0d000a !important; color: #d080c8 !important; border: 1px solid #e8609f !important; border-radius: 10px !important; }
details, .gr-accordion { background: #150018 !important; border: 1px solid #8800cc !important; border-radius: 12px !important; margin-top: 8px !important; }
details summary { color: #f4b8d2 !important; font-weight: 600 !important; padding: 10px !important; cursor: pointer !important; }
.chatbot { background: #0d000a !important; border: 1px solid #c040b0 !important; border-radius: 18px !important; }
select { background: #150018 !important; color: #fde8f0 !important; border: 1.5px solid #e8609f !important; border-radius: 8px !important; }
.status-bar { background: #150018 !important; border: 1px solid #600060 !important; border-radius: 10px !important; padding: 8px 14px !important; color: #f4b8d2 !important; font-size: 12px !important; }
.rel-bar { height: 8px; border-radius: 4px; background: linear-gradient(90deg, #ff1493, #ff85b3); margin: 4px 0; }
::-webkit-scrollbar { width: 5px; }
::-webkit-scrollbar-track { background: #0d000a; }
::-webkit-scrollbar-thumb { background: #9c0f58; border-radius: 3px; }
"""

EXAMPLES_CUSTOM = [
    ["Emma",     "A 24-year-old blonde with ocean-blue eyes and a fit athletic body. A yoga instructor. Playful, confident, endlessly flirty.", "A candlelit rooftop bar at night, city lights glittering below", ""],
    ["Scarlett", "A 27-year-old redhead with emerald green eyes and gorgeous curves. A nurse. Caring on the surface, wild behind closed doors.", "A cozy cabin in the mountains, snow falling outside the window", ""],
    ["Luna",     "A 22-year-old with jet-black hair, pale skin, and mysterious dark eyes. A college student who is shockingly bold and passionate.", "A quiet moonlit library after closing hours", "She once witnessed something she cannot explain. It changed her relationship with reality."],
]

# ─────────────────────────────────────────────────────────────────────────────
# UI
# ─────────────────────────────────────────────────────────────────────────────
with gr.Blocks(
    title="Roleplay Companion v5",
    theme=gr.themes.Soft(
        primary_hue=gr.themes.colors.pink,
        secondary_hue=gr.themes.colors.purple,
        neutral_hue=gr.themes.colors.slate,
        font=[gr.themes.GoogleFont("Nunito"), "Georgia", "serif"],
    ),
    css=CSS, js=JS,
) as demo:

    state_st    = gr.State({})
    memories_st = gr.State([])

    gr.Markdown("# 🌹 Roleplay Companion v5\n*Hermes 3 · Local Prefect Illustrious XL v3 · Direct scene-controlled image prompts*")

    # ── SETUP SCREEN ─────────────────────────────────────────────────────────
    with gr.Group(visible=True) as setup_grp:
        gr.Markdown("### ✨ Create your companion\nPick a scenario preset **or** build your own character below.")

        scenario_dd = gr.Dropdown(
            label="🎭 Scenario Preset (optional — overrides name/desc/scene if selected)",
            choices=list(SCENARIO_PRESETS.keys()),
            value="Custom",
        )

        with gr.Row():
            with gr.Column(scale=2):
                cname_in = gr.Textbox(label="Her Name", placeholder="Emma, Scarlett...", max_lines=1)
                cdesc_in = gr.Textbox(
                    label="Description (appearance FIRST, then personality)",
                    placeholder="A 25-year-old blonde with blue eyes... playful, confident...",
                    lines=3,
                )
                scene_in = gr.Textbox(
                    label="Scene / Setting",
                    placeholder="A candlelit rooftop bar at night...",
                    lines=2,
                )
            with gr.Column(scale=1):
                backstory_in = gr.Textbox(
                    label="🔒 Hidden Backstory (lore the character knows but never announces)",
                    placeholder="She lost her sister three years ago. She never talks about it but it surfaces in quiet moments...",
                    lines=4,
                )

        with gr.Accordion("👥 Optional: Add a Second Character", open=False):
            gr.Markdown("*Two companions will share the same chat and interact with each other and you.*")
            with gr.Row():
                char2_name_in = gr.Textbox(label="Second Character Name", placeholder="Jade, Chloe...", max_lines=1)
                char2_desc_in = gr.Textbox(label="Second Character Description", placeholder="A 26-year-old...", lines=2)

        gr.Markdown(f"**🖼️ Local Image Model:** `{SELECTED_IMAGE_NAME}` selected in Cell 4")
        img_style_setup = gr.Textbox(value=SELECTED_IMAGE_STYLE, visible=False)

        gr.Markdown("**Quick start presets:**")
        gr.Examples(label="", examples=EXAMPLES_CUSTOM, inputs=[cname_in, cdesc_in, scene_in, backstory_in])

        start_btn = gr.Button("💕 Start Chatting", elem_classes="start-btn", variant="primary")

    # ── CHAT SCREEN ──────────────────────────────────────────────────────────
    with gr.Group(visible=False) as chat_grp:

        status_md = gr.Markdown("", elem_classes="status-bar")

        chatbot = gr.Chatbot(
            label="", type="messages", height=500,
            show_label=False, avatar_images=(None, None),
        )

        # Mood row
        gr.Markdown("**🎭 Mood Control:**")
        with gr.Row():
            mood_romantic = gr.Button("🌹 Romantic",   elem_classes="mood-romantic",   scale=1)
            mood_auto     = gr.Button("⚡ Auto",        elem_classes="mood-auto",        scale=1)
            mood_0        = gr.Button("😘 Flirty",      elem_classes="mood-flirty",      scale=1)
            mood_1        = gr.Button("🔥 Suggestive",  elem_classes="mood-suggestive",  scale=1)
            mood_2        = gr.Button("💋 Intimate",    elem_classes="mood-intimate",    scale=1)
            mood_3        = gr.Button("🌶️ Explicit",    elem_classes="mood-explicit",    scale=1)

        # Message input
        with gr.Row():
            msg_in = gr.Textbox(
                label="", show_label=False, lines=2, scale=8,
                placeholder="Type here… Enter = send | Shift+Enter = new line | /pic = photo",
                elem_id="msg_box",
            )
            send_btn = gr.Button("Send", elem_classes="send-btn", scale=1, min_width=70)

        # Action row 1
        with gr.Row():
            img_model_label = gr.Textbox(label="Local Image Model", value=SELECTED_IMAGE_NAME, interactive=False, scale=2)
            img_style_chat = gr.Textbox(value=SELECTED_IMAGE_STYLE, visible=False)
            photo_btn    = gr.Button("📷 Photo",        elem_classes="photo-btn",    scale=2)
            narrator_btn = gr.Button("🎬 Narrator",     elem_classes="narrator-btn", scale=2)
            retry_btn    = gr.Button("♻️ Retry",         elem_classes="retry-btn",    scale=2)

        # Action row 2
        with gr.Row():
            thumbs_up   = gr.Button("👍 More like this",  scale=2)
            thumbs_down = gr.Button("👎 Less like this",  scale=2)
            reset_btn   = gr.Button("🔄 New Chat",        elem_classes="reset-btn", scale=2)
            back_btn    = gr.Button("← Change Character", elem_classes="back-btn",  scale=2)

        # Memory panel
        with gr.Accordion("📌 Pin a Memory", open=False):
            gr.Markdown("*Key moments injected into every prompt so the AI actually remembers them.*")
            with gr.Row():
                memory_in  = gr.Textbox(label="", placeholder='e.g. "she loves coffee" or "we kissed on the balcony"', scale=5)
                pin_btn    = gr.Button("📌 Pin", scale=1)
            memories_display = gr.Textbox(label="Pinned memories", lines=3, interactive=False)

        # Save / Load
        with gr.Accordion("💾 Save / Load Session", open=False):
            gr.Markdown("*Export your conversation as JSON. Reload it later to continue exactly where you left off.*")
            with gr.Row():
                save_btn     = gr.Button("💾 Save Session", scale=2)
                save_status  = gr.Textbox(label="", scale=4, interactive=False)
            with gr.Row():
                load_file    = gr.File(label="Load session JSON", file_types=[".json"], scale=3)
                load_btn     = gr.Button("📂 Load", scale=1)
                load_status  = gr.Textbox(label="", scale=3, interactive=False)

        # Guide
        with gr.Accordion("📖 Guide", open=False):
            gr.Markdown("""
| Turns | Level | Vibe | Photos |
|-------|-------|------|--------|
| Manual | Romantic | Tender, loving | Elegant clothed |
| 0–4 | Flirty | Playful, teasing | Clothed |
| 5–10 | Suggestive | Daring, revealing | Lingerie |
| 11–17 | Intimate | Sensual, nsfw | Topless/nude |
| 18+ | Explicit | Fully uncensored | Explicit |

**Images:** Local Diffusers on T4 using Prefect Illustrious XL v3 SDXL. First image loads the model into GPU VRAM, later images are faster.
**📌 Pin memories:** moments the AI will always remember.
**🎬 Narrator:** generates a cinematic novel passage.
**♻️ Retry:** regenerates the last response with a new seed.
**👍 / 👎:** steers the writing style going forward.
**💾 Save/Load:** export JSON → reload to continue later.
**Auto-summary:** history compresses silently after {threshold} messages.
            """.format(threshold=SUMMARY_THRESHOLD))

    # ── WIRING ───────────────────────────────────────────────────────────────
    start_btn.click(
        fn=do_start,
        inputs=[scenario_dd, cname_in, cdesc_in, scene_in, backstory_in,
                char2_name_in, char2_desc_in, img_style_setup],
        outputs=[setup_grp, chat_grp, chatbot, state_st, status_md],
    )

    def _send(msg, cb, st, sty):
        return do_chat(msg, cb, st, sty)

    send_btn.click(fn=_send, inputs=[msg_in, chatbot, state_st, img_style_chat],
                   outputs=[chatbot, state_st, msg_in, status_md])
    msg_in.submit(fn=_send, inputs=[msg_in, chatbot, state_st, img_style_chat],
                  outputs=[chatbot, state_st, msg_in, status_md])

    photo_btn.click(fn=do_force_photo, inputs=[chatbot, state_st, img_style_chat],
                    outputs=[chatbot, state_st])
    narrator_btn.click(fn=do_narrator, inputs=[chatbot, state_st],
                       outputs=[chatbot, state_st])
    retry_btn.click(fn=do_retry, inputs=[chatbot, state_st, img_style_chat],
                    outputs=[chatbot, state_st, status_md])

    thumbs_up.click(fn=lambda cb, st: do_reaction(cb, st, True),
                    inputs=[chatbot, state_st], outputs=[chatbot, state_st])
    thumbs_down.click(fn=lambda cb, st: do_reaction(cb, st, False),
                      inputs=[chatbot, state_st], outputs=[chatbot, state_st])

    mood_romantic.click(fn=lambda cb, st: do_set_mood(-1, cb, st), inputs=[chatbot, state_st], outputs=[state_st, chatbot])
    mood_0.click(fn=lambda cb, st: do_set_mood(0, cb, st),  inputs=[chatbot, state_st], outputs=[state_st, chatbot])
    mood_1.click(fn=lambda cb, st: do_set_mood(1, cb, st),  inputs=[chatbot, state_st], outputs=[state_st, chatbot])
    mood_2.click(fn=lambda cb, st: do_set_mood(2, cb, st),  inputs=[chatbot, state_st], outputs=[state_st, chatbot])
    mood_3.click(fn=lambda cb, st: do_set_mood(3, cb, st),  inputs=[chatbot, state_st], outputs=[state_st, chatbot])
    mood_auto.click(fn=lambda cb, st: do_auto_mood(cb, st), inputs=[chatbot, state_st], outputs=[state_st, chatbot])

    pin_btn.click(fn=do_pin_memory,
                  inputs=[memory_in, memories_st, chatbot],
                  outputs=[memories_st, chatbot, memory_in])

    def _sync_memories(mems):
        return "\n".join(f"• {m}" for m in mems) if mems else "(none pinned yet)"

    memories_st.change(fn=_sync_memories, inputs=[memories_st], outputs=[memories_display])

    # Keep memories in state
    def _merge_memories(state, mems):
        state = dict(state)
        state["memories"] = list(mems)
        return state
    pin_btn.click(fn=_merge_memories, inputs=[state_st, memories_st], outputs=[state_st])

    save_btn.click(fn=do_save, inputs=[chatbot, state_st], outputs=[load_file, save_status])
    load_btn.click(fn=do_load, inputs=[load_file, chatbot, state_st],
                   outputs=[chatbot, state_st, load_status])

    reset_btn.click(fn=do_reset, inputs=[state_st],
                    outputs=[chatbot, state_st, status_md])
    back_btn.click(fn=do_back,
                   outputs=[setup_grp, chat_grp, chatbot, state_st, status_md])

print("🌹 Launching Roleplay Companion v5...")
demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)






